# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaivarshini-06/flyrank-ml-internship-jaivarshini/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load the anonymized dataset from the repository
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Jaivarshini-06/flyrank-ml-internship-jaivarshini/"
    "main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(DATA_URL)

# Required columns for preparing the feature vector
required = [
    "content_id", "client_id", "impressions_90d",
    "sessions_90d", "content_age_days", "trend_direction"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Define the model features exactly as in scripts/ml_utils.py
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

# Convert numeric values and replace infinities with missing values
for col in numeric_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    else:
        df[col] = np.nan

# Fill missing categorical values consistently
for col in categorical_features:
    if col not in df.columns:
        df[col] = "unknown"
    df[col] = df[col].fillna("unknown").astype(str)
    df[col] = df[col].replace({"": "unknown", "nan": "unknown"})

# Create engineered features used by the model
for col in [
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

# Keep eligible content and one row per content item
df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Define the target separately; never use it as a feature
y = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

X = df[numeric_features + categorical_features].copy()
X[numeric_features] = X[numeric_features].replace(
    [np.inf, -np.inf], np.nan
)

# Numeric missing values are filled with medians; categories use most frequent
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit preprocessing for this feature audit
X_vector = preprocessor.fit_transform(X)

print("Rows prepared:", len(X))
print("Original numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Transformed feature-vector shape:", X_vector.shape)
print("Target positive rate:", round(y.mean(), 4))
print("Missing target values:", int(y.isna().sum()))
print("Target excluded from X:", "trend_direction" not in X.columns)
print("Feature vector built successfully.")



Rows prepared: 30000
Original numeric features: 18
Categorical features: 8
Transformed feature-vector shape: (30000, 52)
Target positive rate: 0.5421
Missing target values: 0
Target excluded from X: True
Feature vector built successfully.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Numeric features

- `search_volume`, `competition`, `cpc`: search demand, competition, and cost-per-click indicators. Numeric conversion is applied; missing values are imputed with the training-set median during model preprocessing.
- `word_count`, `char_count`: content length measures. Missing values are median-imputed.
- `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`: log-transformed performance counts over 90 days, reducing the influence of very large values. The raw counts are converted to numeric values, missing counts are filled with zero, and `log1p` is applied.
- `days_with_impressions`, `days_with_sessions`: number of days with measurable activity. Missing values are median-imputed.
- `content_age_days`, `days_since_last_update`: content age and time since the last update. Missing values are median-imputed.
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`: performance and engagement indicators. Missing values are median-imputed; infinite values are treated as missing.

### Categorical features

- `competition_level`: category describing competition.
- `content_type`: type of content.
- `main_intent`: intended search purpose.
- `age_tier`: content-age category.
- `freshness_tier`: content freshness category.
- `word_count_tier`: content-length category based on word count.
- `impression_tier`: category representing impressions.
- `position_tier`: category representing search position.

Missing categorical values are filled with `unknown` before encoding. One-hot encoding converts categories into numeric columns, and unseen categories are ignored by the encoder.

### Availability and timing

These features should only be used if their values are known at the time a prediction is made. The 90-day metrics and derived tiers must refer to a completed historical window, not a window containing future outcomes. The dataset's exact prediction cutoff must be verified before claiming that every feature is available beforehand.

The target is derived from `trend_direction == "down"`. `trend_direction` is excluded from the feature vector because it directly defines the target. The related `trend_pct` field is also excluded because it can reveal the outcome.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.